[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Lucas-Maingi/seer/blob/main/notebooks/train_colab.ipynb)

# Seer — cloud training run

Trains all three Seer models on a free GPU (Colab T4 / Kaggle) and hands back
a small zip of deployable ONNX weights for the CPU serving machine.

**Runtime → Change runtime type → T4 GPU** before running.

**Resilient to disconnects.** Every stage below checks Google Drive for a
prior result before doing any work, and backs up its own result to Drive as
soon as it finishes. If your session drops (free Colab reclaims idle/long-
running VMs, laptop sleeps, etc.), just reopen this notebook, connect a GPU
runtime again, and **Run all** — finished stages are skipped, and only the
stage that was in flight when you got cut off needs to redo its work.

Everything trained here is synthetic specimen data generated in this
session — nothing sensitive is uploaded or downloaded.

In [ ]:
!nvidia-smi -L

## Mount Drive (the resume point)

Everything that survives a disconnect lives under
`/content/drive/MyDrive/seer_backup/`. Authorize access when prompted.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
BACKUP = "/content/drive/MyDrive/seer_backup"
os.makedirs(BACKUP, exist_ok=True)
print("backup dir:", BACKUP)

In [ ]:
import os
if not os.path.isdir("/content/seer"):
    !git clone https://github.com/Lucas-Maingi/seer.git /content/seer
%cd /content/seer
!pip install -q -e .
os.makedirs("data", exist_ok=True)
os.makedirs("weights", exist_ok=True)
os.makedirs("runs", exist_ok=True)

In [ ]:
# Restore whatever a previous (interrupted) run already finished.
import os, shutil

def restore(name, dest):
    src = f"{BACKUP}/{name}"
    if os.path.exists(src) and not os.path.exists(dest):
        print(f"restoring {name} from Drive")
        (shutil.copytree if os.path.isdir(src) else shutil.copy2)(src, dest)

restore("synth", "data/synth")
for f in ("corner.pt", "crnn.pt", "tamper.pt"):
    restore(f, f"weights/{f}")

print("dataset present:", os.path.exists("data/synth/index.jsonl"))
print("checkpoints present:",
      [f for f in ("corner.pt", "crnn.pt", "tamper.pt") if os.path.exists(f"weights/{f}")])

## 1/6 — Generate the synthetic dataset

~8k samples is a good balance for models this small (raise `COUNT` if you have
session time to spare — generation is CPU-bound, ~1.4 s/sample, ~1h45m at 8k).
Skipped automatically if a dataset already exists locally or in the Drive
backup.

Optional realism upgrade for the portraits: upload a folder of synthetic faces
(e.g. a slice of the [SFHQ dataset](https://github.com/SelfishGene/SFHQ-dataset))
and pass `FACES=/content/faces`.

In [ ]:
import os
os.environ.setdefault("COUNT", "8000")
faces_flag = f'--faces {os.environ["FACES"]}' if os.environ.get("FACES") else ""

if not os.path.exists("data/synth/index.jsonl"):
    !python -m seer.synth.generate --out data/synth --count $COUNT $faces_flag
    !cp -r data/synth "$BACKUP/synth"
    print("dataset generated and backed up to Drive")
else:
    print("dataset already present, skipping generation")

## 2/6 — Corner localization

In [ ]:
import os
if not os.path.exists("weights/corner.pt"):
    !python -m seer.localize.train --data data/synth --epochs "${EPOCHS_CORNER:-30}" --out weights/corner.pt
    !cp weights/corner.pt "$BACKUP/corner.pt"
    print("corner model trained and backed up to Drive")
else:
    print("corner model already trained, skipping")

## 3/6 — Field OCR

In [ ]:
import os
if not os.path.exists("weights/crnn.pt"):
    !python -m seer.ocr.train --data data/synth --epochs "${EPOCHS_CRNN:-40}" --out weights/crnn.pt
    !cp weights/crnn.pt "$BACKUP/crnn.pt"
    print("OCR model trained and backed up to Drive")
else:
    print("OCR model already trained, skipping")

## 4/6 — Tamper forensics

In [ ]:
import os
if not os.path.exists("weights/tamper.pt"):
    !python -m seer.forensics.train --data data/synth --epochs "${EPOCHS_TAMPER:-25}" --out weights/tamper.pt
    !cp weights/tamper.pt "$BACKUP/tamper.pt"
    print("tamper model trained and backed up to Drive")
else:
    print("tamper model already trained, skipping")

## 5/6 — ONNX export + INT8 quantization

Cheap (seconds to a couple minutes) — always safe to re-run.

In [ ]:
!python -m seer.runtime.export --all
!python -m seer.runtime.quantize --all --calib data/synth
!cp weights/*.onnx "$BACKUP/" 2>/dev/null || true
print("ONNX weights exported, quantized, and backed up to Drive")

## 6/6 — Latency benchmark (indicative only — re-run on the serving CPU)

In [ ]:
!python -m seer.runtime.bench --report runs/latency.md
!cp runs/latency.md "$BACKUP/latency.md" 2>/dev/null || true

## Package the deployable weights

Only the ONNX files (and training checkpoints if you want to resume later)
need to leave this machine — a few tens of MB.

In [ ]:
!zip -j seer_weights.zip weights/*.onnx weights/*.pt runs/latency.md
from google.colab import files
files.download("seer_weights.zip")

## Back on the laptop

```bash
unzip seer_weights.zip -d weights/
python scripts/fetch_models.py          # YuNet + ArcFace for the face stage
python -m seer.runtime.bench            # re-bench on the real serving CPU
uvicorn seer.api.main:app --port 8000   # and in web/: npm run dev
```

The latency report that matters is the one measured on the serving hardware,
not this VM — rerun `seer.runtime.bench` locally and commit that.